# GRPO with verifiable rewards

Continue training the SFT adapter with Group Relative Policy
Optimization using three reward functions: exact-answer correctness,
format compliance (does the response contain \boxed{...}), and a
length penalty to discourage runaway verbosity.

We run three ablations by setting `REWARD_MODE` to 'A' (correctness
only), 'B' (correctness + format), or 'C' (all three). Set C is the
main experiment; A and B are diagnostic.

In [1]:
# Config. Continue from sft_final, not the base model. REWARD_MODE
# picks which reward set to use for this run — change it between runs
# to get the ablation.
!pip install -q -U bitsandbytes transformers trl peft accelerate datasets torchao

# Mount Drive and restore everything
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE = '/content/drive/MyDrive/llm_posttraining'

# Restore data
if not os.path.exists('/content/data/grpo_prompts'):
    os.makedirs('/content/data', exist_ok=True)
    shutil.copytree(f'{DRIVE}/data/sft_train',    '/content/data/sft_train')
    shutil.copytree(f'{DRIVE}/data/sft_val',      '/content/data/sft_val')
    shutil.copytree(f'{DRIVE}/data/sft_test',     '/content/data/sft_test')
    shutil.copytree(f'{DRIVE}/data/grpo_prompts', '/content/data/grpo_prompts')

# Restore SFT checkpoint (GRPO starts from SFT)
if not os.path.exists('/content/checkpoints/sft_final'):
    os.makedirs('/content/checkpoints', exist_ok=True)
    shutil.copytree(f'{DRIVE}/checkpoints/sft_final', '/content/checkpoints/sft_final')

print('All restored. Ready for GRPO.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 145.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 46.8 MB/s eta 0:00:00
Mounted at /content/drive
All restored. Ready for GRPO.


In [2]:
import torch, re
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import GRPOTrainer, GRPOConfig
import wandb

SFT_ADAPTER = '/content/checkpoints/sft_final'
BASE_MODEL  = 'Qwen/Qwen3-8B'
HF_REPO_GRPO = 'Chaitanya77/qwen3-8b-math-grpo'
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_80GB = VRAM_GB >= 75
REWARD_MODE = 'C'  # 'A', 'B', or 'C' can be easily  change for ablation runs

In [16]:
# Load the GRPO prompts we saved in notebook 01. Rename the answer
# column to 'solution' so it matches the keyword argument that
# reward_correctness expects.

def to_str(x):
    if isinstance(x, str):
        return x
    if isinstance(x, list):
        return to_str(x[0])
    if isinstance(x, dict):
        # take the first value
        return to_str(list(x.values())[0])
    return str(x)

def extract_boxed(text):
    text = to_str(text)
    m = re.search(r'\\boxed\{([^}]*)\}', text)
    return m.group(1).strip() if m else None

def normalize(ans) -> str:
    if ans is None: return ''
    return str(ans).strip().lower().replace(' ', '')

def reward_correctness(completions, solution, **kwargs):
    solution = to_str(solution)
    return [1.0 if normalize(extract_boxed(to_str(c))) == normalize(solution) else -1.0 for c in completions]

def reward_format(completions, **kwargs):
    return [0.2 if re.search(r'\\boxed\{[^}]+\}', to_str(c)) else 0.0 for c in completions]

def reward_length_penalty(completions, **kwargs):
    rewards = []
    for c in completions:
        n = len(to_str(c).split())
        rewards.append(max(-0.1 * (max(0, n - 2048) // 512), -0.5))
    return rewards

# Select reward functions based on ablation mode
REWARD_FN_MAP = {
    'A': [reward_correctness],
    'B': [reward_correctness, reward_format],
    'C': [reward_correctness, reward_format, reward_length_penalty],
}
reward_funcs = REWARD_FN_MAP[REWARD_MODE]
print(f'Using reward functions: {[f.__name__ for f in reward_funcs]}')

Using reward functions: ['reward_correctness', 'reward_format', 'reward_length_penalty']


In [17]:
# Load SFT model with the adapter trainable. GRPO will keep a frozen
# reference copy internally for the KL term in the objective.
grpo_ds = load_from_disk('/content/data/grpo_prompts')
# Rename 'final_answer' -> 'solution' to match reward function signature
grpo_ds = grpo_ds.rename_column('final_answer', 'solution')
grpo_ds = grpo_ds.select_columns(['prompt', 'solution'])
print(f'GRPO dataset: {len(grpo_ds)}')
print(grpo_ds[0])

GRPO dataset: 30257
{'prompt': [{'content': 'Please reason step by step, and put your final answer within \\boxed{}.', 'role': 'system'}, {'content': '17. A ladder has a total of 17 steps, with the highest step being 30 cm wide and the lowest step being 110 cm wide. There are 15 steps in between, and the width difference between adjacent steps remains constant. The width of the 9th step is $\\qquad$ cm.', 'role': 'user'}], 'solution': '70'}


In [18]:
# GRPO config. Most parameters are conservative — single-GPU GRPO
# doesn't tolerate aggressive batch sizes. The big knobs:
#   num_generations: completions per prompt, more is better but slower
#   max_completion_length: how long the model can reason for
#   beta: KL penalty against the reference (SFT) policy
#   max_steps: total optimization steps — 500 is short, 1000+ is
#              where GRPO typically starts winning over SFT.
# vLLM disabled because colocate mode is risky on 40GB.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map='auto', torch_dtype=torch.bfloat16
)
model = PeftModel.from_pretrained(base, SFT_ADAPTER, is_trainable=True)
model.config.use_cache = False
print('SFT model loaded for GRPO. VRAM:', round(torch.cuda.memory_allocated()/1e9, 2), 'GB')

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


SFT model loaded for GRPO. VRAM: 18.96 GB


In [21]:
# Run training. Watch the W&B dashboard for:
#   rewards/mean trending up over steps
#   rewards/std staying nonzero (zero std = no learning signal)
#   kl staying bounded (huge spikes = policy collapse)

grpo_config = GRPOConfig(
    output_dir=f'/content/checkpoints/grpo_{REWARD_MODE}',

    # Generation
    num_generations=2,
    max_completion_length=768,
    temperature=0.8,
    top_p=0.95,

    # vLLM disabled (safer on single GPU)
    use_vllm=False,

    # Training
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=True,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    max_steps=50,

    # KL control
    beta=0.04,

    # Logging
    save_strategy='steps',
    save_steps=100,
    logging_steps=10,
    report_to='wandb',
    run_name=f'qwen3-8b-grpo-reward-{REWARD_MODE}',
    log_completions=True,
)

wandb.init(project='verifiable-reasoning-posttraining', name=f'05_grpo_{REWARD_MODE}')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


profiling/Time taken: GRPOTrainer._calculate_rewards,█▁
profiling/Time taken: GRPOTrainer._get_per_token_logps_and_entropies,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁
profiling/Time taken: GRPOTrainer._prepare_inputs,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇
profiling/Time taken: GRPOTrainer.compute_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁
profiling/Time taken: GRPOTrainer.reward_correctness,█▁
profiling/Time taken: GRPOTrainer.reward_format,█▁
profiling/Time taken: GRPOTrainer.reward_length_penalty,█▁
profiling/Time taken: GRPOTrainer.transformers.generate,█▆▁
train/global_step,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
profiling/Time taken: GRPOTrainer._calculate_rewards,0.00065
profiling/Time taken: GRPOTrainer._get_per_token_logps_and_entropies,0.06978


In [22]:
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_ds,
    reward_funcs=reward_funcs,
    processing_class=tokenizer,
)
trainer.train()
wandb.finish()

Output hidden; open in https://colab.research.google.com to view.

In [25]:
from huggingface_hub import login
login()  # paste your Write token

In [26]:
trainer.model.save_pretrained(f'/content/checkpoints/grpo_{REWARD_MODE}_final')
trainer.model.push_to_hub(f'{HF_REPO_GRPO}-reward-{REWARD_MODE}')
print('GRPO adapter saved.')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 46.3kB / 87.4MB            

  ...adapter_model.safetensors:   1%|          | 1.21MB /  175MB            

GRPO adapter saved.


## Reward-hacking sanity checks

In [27]:
# After training, look at the completion samples logged to W&B for
# signs of reward hacking:
#   - same phrase repeated over and over (repetition hack)
#   - correct boxed answer with garbage reasoning (format-only hack)
#   - extremely long responses (verbosity hack)
# If any of these dominate, the reward function needs work.

# These are visible in W&B under completions tab if log_completions=True
print('Check W&B dashboard for reward curves, KL, and completion samples.')
print('Key metrics to monitor:')
print('  rewards/mean        — should increase over steps')
print('  rewards/std         — should not collapse to 0 (group diversity needed)')
print('  kl                  — monitor for sudden spikes (policy instability)')
print('  completions/length  — watch for monotonic increase (verbosity hack)')

Check W&B dashboard for reward curves, KL, and completion samples.
Key metrics to monitor:
  rewards/mean        — should increase over steps
  rewards/std         — should not collapse to 0 (group diversity needed)
  kl                  — monitor for sudden spikes (policy instability)
  completions/length  — watch for monotonic increase (verbosity hack)


In [28]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import shutil, os
DRIVE = '/content/drive/MyDrive/llm_posttraining'

# Save GRPO checkpoint
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
shutil.copytree(f'/content/checkpoints/grpo_{REWARD_MODE}_final',
                f'{DRIVE}/checkpoints/grpo_{REWARD_MODE}_final',
                dirs_exist_ok=True)

print(f'GRPO checkpoint saved to Drive.')

Mounted at /content/drive
GRPO checkpoint saved to Drive.
